# Confidence-Only Monitor

This notebook is a strict baseline that uses only the logged `confidence`
field to predict downstream success. It keeps the same file split,
sampling rule, and evaluation protocol as the main monitor so the results
are directly comparable.

Baseline definition:
- train only on mixed-root files
- validate/test only on pure-root files
- sample prefixes with the same `stride = 8` and `min_step = 9`
- use the current row's `confidence` as the only signal
- fit a 1D logistic calibrator on top of imputed confidence
- choose the alarm threshold on validation, then report test metrics


In [1]:
from __future__ import annotations

import csv
import json
import math
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -35.0, 35.0)
    return 1.0 / (1.0 + np.exp(-x))


def safe_float(value: str) -> float:
    if value is None or value == "":
        return 0.0
    if value.lower() == "nan":
        return float("nan")
    return float(value)


def safe_int(value: str) -> int:
    if value is None or value == "":
        return 0
    return int(float(value))


def is_pure_zero(y: float, tol: float = 1e-9) -> bool:
    return abs(y - 0.0) <= tol


def is_pure_one(y: float, tol: float = 1e-9) -> bool:
    return abs(y - 1.0) <= tol


def root_kind_from_y(y: float) -> str:
    if is_pure_zero(y):
        return "pure0"
    if is_pure_one(y):
        return "pure1"
    return "mixed"


def node_parent(node_id: str) -> str | None:
    if node_id == "0":
        return None
    return node_id.rsplit(".", 1)[0]


def node_depth(node_id: str) -> int:
    return node_id.count(".")


def sorted_node_ids(node_ids: Iterable[str]) -> List[str]:
    def key(node_id: str) -> Tuple[int, List[int]]:
        if node_id == "0":
            return (0, [0])
        return (node_depth(node_id), [int(part) for part in node_id.split(".")])

    return sorted(node_ids, key=key)


@dataclass
class RowRecord:
    instance_id: str
    node_id: str
    step_idx: int
    confidence: float
    y: float


@dataclass
class InstanceData:
    instance_id: str
    root_y: float
    root_kind: str
    rows_by_node: Dict[str, List[RowRecord]]
    children_by_parent: Dict[str, List[str]]


@dataclass
class PrefixSample:
    instance_id: str
    root_kind: str
    node_id: str
    step_idx: int
    target_y: float
    confidence: float
    is_final_step: bool
    is_root_prefix: bool


@dataclass
class SplitSpec:
    train_mixed: List[str]
    val_pure: List[str]
    test_pure: List[str]


@dataclass
class DatasetBundle:
    train_samples: List[PrefixSample]
    val_samples: List[PrefixSample]
    test_samples: List[PrefixSample]
    splits: SplitSpec


class LogisticCalibrator:
    def __init__(self, learning_rate: float, epochs: int, l2: float) -> None:
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2 = l2
        self.scale = 1.0
        self.bias = 0.0
        self.best_loss = float("inf")
        self.history: List[Dict[str, float]] = []

    def fit(self, scores: np.ndarray, targets: np.ndarray) -> "LogisticCalibrator":
        scores = np.asarray(scores, dtype=np.float64).reshape(-1)
        targets = np.asarray(targets, dtype=np.float64).reshape(-1)

        m_scale = 0.0
        v_scale = 0.0
        m_bias = 0.0
        v_bias = 0.0
        beta1 = 0.9
        beta2 = 0.999
        eps = 1e-8

        best_scale = self.scale
        best_bias = self.bias

        for epoch in range(1, self.epochs + 1):
            logits = self.scale * scores + self.bias
            preds = sigmoid(logits)
            residual = preds - targets

            grad_scale = float(np.mean(residual * scores) + self.l2 * self.scale)
            grad_bias = float(np.mean(residual))
            clipped = np.clip(preds, 1e-8, 1.0 - 1e-8)
            loss = float(
                -np.mean(targets * np.log(clipped) + (1.0 - targets) * np.log(1.0 - clipped))
                + 0.5 * self.l2 * self.scale * self.scale
            )
            self.history.append({"epoch": float(epoch), "loss": float(loss)})

            if loss < self.best_loss:
                self.best_loss = float(loss)
                best_scale = self.scale
                best_bias = self.bias

            m_scale = beta1 * m_scale + (1.0 - beta1) * grad_scale
            v_scale = beta2 * v_scale + (1.0 - beta2) * (grad_scale ** 2)
            m_bias = beta1 * m_bias + (1.0 - beta1) * grad_bias
            v_bias = beta2 * v_bias + (1.0 - beta2) * (grad_bias ** 2)

            m_scale_hat = m_scale / (1.0 - beta1**epoch)
            v_scale_hat = v_scale / (1.0 - beta2**epoch)
            m_bias_hat = m_bias / (1.0 - beta1**epoch)
            v_bias_hat = v_bias / (1.0 - beta2**epoch)

            self.scale -= self.learning_rate * m_scale_hat / (math.sqrt(v_scale_hat) + eps)
            self.bias -= self.learning_rate * m_bias_hat / (math.sqrt(v_bias_hat) + eps)

        self.scale = best_scale
        self.bias = best_bias
        return self

    def predict_success_prob(self, scores: np.ndarray) -> np.ndarray:
        scores = np.asarray(scores, dtype=np.float64)
        return sigmoid(self.scale * scores + self.bias)

    def to_dict(self) -> Dict[str, float]:
        return {
            "scale": float(self.scale),
            "bias": float(self.bias),
            "best_loss": float(self.best_loss),
        }


def load_instance(csv_path: Path) -> InstanceData:
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        rows = []
        for row in reader:
            rows.append(
                RowRecord(
                    instance_id=row["instance_id"],
                    node_id=row["node_id"],
                    step_idx=safe_int(row["step_idx"]),
                    confidence=safe_float(row["confidence"]),
                    y=safe_float(row["y"]),
                )
            )

    if not rows:
        raise ValueError(f"No rows found in {csv_path}")

    instance_id = rows[0].instance_id
    root_y = rows[0].y
    root_kind = root_kind_from_y(root_y)
    rows_by_node: Dict[str, List[RowRecord]] = defaultdict(list)
    for row in rows:
        rows_by_node[row.node_id].append(row)
    for node_rows in rows_by_node.values():
        node_rows.sort(key=lambda item: item.step_idx)

    children_by_parent: Dict[str, List[str]] = defaultdict(list)
    for node_id in rows_by_node:
        parent = node_parent(node_id)
        if parent is not None:
            children_by_parent[parent].append(node_id)
    for parent, children in children_by_parent.items():
        children_by_parent[parent] = sorted_node_ids(children)

    return InstanceData(
        instance_id=instance_id,
        root_y=root_y,
        root_kind=root_kind,
        rows_by_node=dict(rows_by_node),
        children_by_parent=dict(children_by_parent),
    )


def load_all_instances(data_dir: Path) -> Dict[str, InstanceData]:
    instances: Dict[str, InstanceData] = {}
    for csv_path in sorted(data_dir.glob("*.csv")):
        instance = load_instance(csv_path)
        instances[instance.instance_id] = instance
    return instances


def stratified_pure_split(
    instances: Dict[str, InstanceData],
    pure_val_ratio: float,
    seed: int,
) -> SplitSpec:
    rng = random.Random(seed)
    pure0 = [inst_id for inst_id, inst in instances.items() if inst.root_kind == "pure0"]
    pure1 = [inst_id for inst_id, inst in instances.items() if inst.root_kind == "pure1"]
    mixed = [inst_id for inst_id, inst in instances.items() if inst.root_kind == "mixed"]

    rng.shuffle(pure0)
    rng.shuffle(pure1)

    def split_half(ids: List[str]) -> Tuple[List[str], List[str]]:
        cut = int(round(len(ids) * pure_val_ratio))
        cut = min(max(cut, 1), max(len(ids) - 1, 1)) if len(ids) > 1 else len(ids)
        return ids[:cut], ids[cut:]

    val0, test0 = split_half(pure0)
    val1, test1 = split_half(pure1)

    return SplitSpec(
        train_mixed=sorted(mixed),
        val_pure=sorted(val0 + val1),
        test_pure=sorted(test0 + test1),
    )


def should_sample_step(
    step_idx: int,
    node_step_indices: Sequence[int],
    stride: int,
    min_step: int,
) -> bool:
    if step_idx < min_step:
        return False
    return (step_idx - 1) % stride == 0 or step_idx == node_step_indices[-1]


def build_samples(
    instances: Dict[str, InstanceData],
    split_spec: SplitSpec,
    stride: int,
    min_step: int,
) -> DatasetBundle:
    train_samples: List[PrefixSample] = []
    val_samples: List[PrefixSample] = []
    test_samples: List[PrefixSample] = []

    train_set = set(split_spec.train_mixed)
    val_set = set(split_spec.val_pure)
    test_set = set(split_spec.test_pure)

    for instance_id, instance in instances.items():
        for node_id in sorted_node_ids(instance.rows_by_node):
            node_rows = instance.rows_by_node[node_id]
            node_step_indices = [row.step_idx for row in node_rows]
            for row in node_rows:
                if not should_sample_step(row.step_idx, node_step_indices, stride, min_step):
                    continue
                sample = PrefixSample(
                    instance_id=instance_id,
                    root_kind=instance.root_kind,
                    node_id=node_id,
                    step_idx=row.step_idx,
                    target_y=row.y,
                    confidence=row.confidence,
                    is_final_step=(row.step_idx == node_step_indices[-1]),
                    is_root_prefix=(node_id == "0"),
                )
                if instance_id in train_set:
                    train_samples.append(sample)
                elif instance_id in val_set:
                    val_samples.append(sample)
                elif instance_id in test_set:
                    test_samples.append(sample)

    return DatasetBundle(
        train_samples=train_samples,
        val_samples=val_samples,
        test_samples=test_samples,
        splits=split_spec,
    )


def confidence_arrays(
    train_samples: Sequence[PrefixSample],
    other_samples: Sequence[PrefixSample],
) -> Tuple[np.ndarray, np.ndarray, float, float, float]:
    train_raw = np.asarray([sample.confidence for sample in train_samples], dtype=np.float64)
    other_raw = np.asarray([sample.confidence for sample in other_samples], dtype=np.float64)

    non_missing = train_raw[~np.isnan(train_raw)]
    fill_value = float(non_missing.mean()) if non_missing.size else 0.5

    train_missing_rate = float(np.isnan(train_raw).mean()) if train_raw.size else 0.0
    other_missing_rate = float(np.isnan(other_raw).mean()) if other_raw.size else 0.0

    train_imputed = np.where(np.isnan(train_raw), fill_value, train_raw)
    other_imputed = np.where(np.isnan(other_raw), fill_value, other_raw)
    train_imputed = np.clip(train_imputed, 0.0, 1.0)
    other_imputed = np.clip(other_imputed, 0.0, 1.0)
    return train_imputed, other_imputed, fill_value, train_missing_rate, other_missing_rate


def roc_auc_score_binary(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_score = np.asarray(y_score, dtype=np.float64)
    positives = int(y_true.sum())
    negatives = int((1 - y_true).sum())
    if positives == 0 or negatives == 0:
        return float("nan")

    order = np.argsort(y_score)
    sorted_scores = y_score[order]
    sorted_labels = y_true[order]

    rank_sum = 0.0
    idx = 0
    n = len(y_true)
    while idx < n:
        end = idx + 1
        while end < n and sorted_scores[end] == sorted_scores[idx]:
            end += 1
        avg_rank = (idx + end - 1) / 2.0 + 1.0
        positive_count = int(sorted_labels[idx:end].sum())
        rank_sum += positive_count * avg_rank
        idx = end

    auc = (rank_sum - positives * (positives + 1) / 2.0) / (positives * negatives)
    return float(auc)


def average_precision_binary(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_score = np.asarray(y_score, dtype=np.float64)
    positives = int(y_true.sum())
    if positives == 0:
        return float("nan")

    order = np.argsort(-y_score)
    y_true = y_true[order]
    tp = 0.0
    fp = 0.0
    ap = 0.0
    for label in y_true:
        if label == 1:
            tp += 1
            ap += tp / max(tp + fp, 1.0)
        else:
            fp += 1
    return float(ap / positives)


def choose_alarm_threshold(y_true_fail: np.ndarray, p_success: np.ndarray) -> Dict[str, float]:
    fail_score = 1.0 - p_success
    thresholds = np.unique(np.round(fail_score, 6))
    if thresholds.size > 400:
        thresholds = np.linspace(float(fail_score.min()), float(fail_score.max()), 400)

    best = None
    for threshold in thresholds:
        pred_fail = (fail_score >= threshold).astype(np.int64)
        tp = int(((pred_fail == 1) & (y_true_fail == 1)).sum())
        tn = int(((pred_fail == 0) & (y_true_fail == 0)).sum())
        fp = int(((pred_fail == 1) & (y_true_fail == 0)).sum())
        fn = int(((pred_fail == 0) & (y_true_fail == 1)).sum())

        recall = tp / max(tp + fn, 1)
        specificity = tn / max(tn + fp, 1)
        precision = tp / max(tp + fp, 1)
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
        bal_acc = 0.5 * (recall + specificity)

        candidate = {
            "threshold": float(threshold),
            "balanced_accuracy": float(bal_acc),
            "f1": float(f1),
            "precision": float(precision),
            "recall": float(recall),
            "specificity": float(specificity),
        }
        if best is None or (
            candidate["balanced_accuracy"],
            candidate["f1"],
            candidate["recall"],
            -candidate["threshold"],
        ) > (
            best["balanced_accuracy"],
            best["f1"],
            best["recall"],
            -best["threshold"],
        ):
            best = candidate
    if best is None:
        raise RuntimeError("Unable to choose threshold")
    return best


def binary_metrics(y_fail: np.ndarray, p_success: np.ndarray, threshold: float) -> Dict[str, float]:
    fail_score = 1.0 - p_success
    pred_fail = (fail_score >= threshold).astype(np.int64)
    y_fail = y_fail.astype(np.int64)

    tp = int(((pred_fail == 1) & (y_fail == 1)).sum())
    tn = int(((pred_fail == 0) & (y_fail == 0)).sum())
    fp = int(((pred_fail == 1) & (y_fail == 0)).sum())
    fn = int(((pred_fail == 0) & (y_fail == 1)).sum())

    accuracy = (tp + tn) / max(len(y_fail), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
    bal_acc = 0.5 * (recall + specificity)
    auc = roc_auc_score_binary(y_fail, fail_score)
    ap = average_precision_binary(y_fail, fail_score)
    brier = float(np.mean((p_success - (1.0 - y_fail)) ** 2))
    clipped = np.clip(p_success, 1e-8, 1.0 - 1e-8)
    log_loss = float(
        -np.mean((1.0 - y_fail) * np.log(clipped) + y_fail * np.log(1.0 - clipped))
    )

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(bal_acc),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "roc_auc": float(auc),
        "average_precision": float(ap),
        "brier": float(brier),
        "log_loss": float(log_loss),
        "alarm_rate": float(pred_fail.mean()),
        "tp": float(tp),
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
    }


def trajectory_alarm_metrics(
    samples: Sequence[PrefixSample],
    p_success: np.ndarray,
    threshold: float,
    instances: Dict[str, InstanceData],
) -> Dict[str, float]:
    pred_fail = np.asarray((1.0 - p_success) >= threshold, dtype=np.int64)
    grouped: Dict[str, Dict[str, List[Tuple[int, int]]]] = defaultdict(lambda: defaultdict(list))
    for sample, pred in zip(samples, pred_fail):
        grouped[sample.instance_id][sample.node_id].append((sample.step_idx, int(pred)))

    pure0_total = 0
    pure1_total = 0
    pure0_detected = 0
    pure1_false = 0
    pure0_first_steps: List[int] = []
    pure1_first_steps: List[int] = []
    pure0_early_warning_pct: List[float] = []

    for instance_id, by_node in grouped.items():
        instance = instances[instance_id]
        leaf_nodes = [
            node_id for node_id in instance.rows_by_node if node_id not in instance.children_by_parent
        ]
        for leaf_node in leaf_nodes:
            path_nodes: List[str] = []
            current = leaf_node
            while current is not None:
                path_nodes.append(current)
                current = node_parent(current)
            path_nodes.reverse()

            path_events: List[Tuple[int, int]] = []
            for node_id in path_nodes:
                path_events.extend(by_node.get(node_id, []))
            path_events.sort(key=lambda item: item[0])

            first_alarm = next((step for step, is_alarm in path_events if is_alarm == 1), None)
            final_step = instance.rows_by_node[leaf_node][-1].step_idx

            if instance.root_kind == "pure0":
                pure0_total += 1
                if first_alarm is not None:
                    pure0_detected += 1
                    pure0_first_steps.append(first_alarm)
                    pure0_early_warning_pct.append(
                        100.0 * (final_step - first_alarm) / max(final_step, 1)
                    )
            elif instance.root_kind == "pure1":
                pure1_total += 1
                if first_alarm is not None:
                    pure1_false += 1
                    pure1_first_steps.append(first_alarm)

    return {
        "pure0_leaf_alarm_recall": pure0_detected / max(pure0_total, 1),
        "pure1_leaf_false_alarm_rate": pure1_false / max(pure1_total, 1),
        "pure0_leaf_alarm_median_step": float(np.median(pure0_first_steps))
        if pure0_first_steps
        else float("nan"),
        "pure1_leaf_false_alarm_median_step": float(np.median(pure1_first_steps))
        if pure1_first_steps
        else float("nan"),
        "pure0_leaf_paths": float(pure0_total),
        "pure1_leaf_paths": float(pure1_total),
        "pure0_successful_warning_mean_early_pct": float(np.mean(pure0_early_warning_pct))
        if pure0_early_warning_pct
        else float("nan"),
    }


def save_predictions_csv(
    path: Path,
    samples: Sequence[PrefixSample],
    confidence_imputed: np.ndarray,
    p_success: np.ndarray,
    threshold: float,
) -> None:
    pred_fail = np.asarray((1.0 - p_success) >= threshold, dtype=np.int64)
    fieldnames = [
        "instance_id",
        "root_kind",
        "node_id",
        "step_idx",
        "target_y",
        "raw_confidence",
        "imputed_confidence",
        "pred_success",
        "pred_alarm",
        "pred_alarm_flag",
    ]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for sample, conf, pred, alarm_flag in zip(samples, confidence_imputed, p_success, pred_fail):
            writer.writerow(
                {
                    "instance_id": sample.instance_id,
                    "root_kind": sample.root_kind,
                    "node_id": sample.node_id,
                    "step_idx": sample.step_idx,
                    "target_y": f"{sample.target_y:.6f}",
                    "raw_confidence": "" if math.isnan(sample.confidence) else f"{sample.confidence:.6f}",
                    "imputed_confidence": f"{float(conf):.6f}",
                    "pred_success": f"{float(pred):.6f}",
                    "pred_alarm": f"{1.0 - float(pred):.6f}",
                    "pred_alarm_flag": int(alarm_flag),
                }
            )


def run_experiment(
    data_dir: Path,
    output_dir: Path,
    stride: int = 8,
    min_step: int = 9,
    pure_val_ratio: float = 0.5,
    seed: int = 7,
    calibration_learning_rate: float = 0.1,
    calibration_epochs: int = 400,
    calibration_l2: float = 1e-3,
) -> Dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=True)
    instances = load_all_instances(data_dir)
    if not instances:
        raise RuntimeError(f"No CSV files found under {data_dir}")

    splits = stratified_pure_split(instances, pure_val_ratio, seed)
    bundle = build_samples(
        instances=instances,
        split_spec=splits,
        stride=stride,
        min_step=min_step,
    )

    y_train = np.asarray([sample.target_y for sample in bundle.train_samples], dtype=np.float64)
    y_val_fail = np.asarray([1 if sample.root_kind == "pure0" else 0 for sample in bundle.val_samples])
    y_test_fail = np.asarray([1 if sample.root_kind == "pure0" else 0 for sample in bundle.test_samples])

    train_conf, val_conf, fill_value, train_missing_rate, val_missing_rate = confidence_arrays(
        bundle.train_samples, bundle.val_samples
    )
    _, test_conf, _, _, test_missing_rate = confidence_arrays(bundle.train_samples, bundle.test_samples)

    calibrator = LogisticCalibrator(
        learning_rate=calibration_learning_rate,
        epochs=calibration_epochs,
        l2=calibration_l2,
    ).fit(train_conf, y_train)

    train_p = calibrator.predict_success_prob(train_conf)
    val_p = calibrator.predict_success_prob(val_conf)
    test_p = calibrator.predict_success_prob(test_conf)

    threshold_info = choose_alarm_threshold(y_val_fail, val_p)
    alarm_threshold = threshold_info["threshold"]
    success_threshold = 1.0 - alarm_threshold

    val_metrics = binary_metrics(y_val_fail, val_p, alarm_threshold)
    test_metrics = binary_metrics(y_test_fail, test_p, alarm_threshold)
    val_leaf_metrics = trajectory_alarm_metrics(
        bundle.val_samples, val_p, alarm_threshold, instances
    )
    test_leaf_metrics = trajectory_alarm_metrics(
        bundle.test_samples, test_p, alarm_threshold, instances
    )

    summary = {
        "config": {
            "data_dir": str(data_dir),
            "stride": stride,
            "min_step": min_step,
            "pure_val_ratio": pure_val_ratio,
            "seed": seed,
            "calibration_learning_rate": calibration_learning_rate,
            "calibration_epochs": calibration_epochs,
            "calibration_l2": calibration_l2,
        },
        "counts": {
            "num_instances": len(instances),
            "mixed_train_files": len(bundle.splits.train_mixed),
            "val_pure_files": len(bundle.splits.val_pure),
            "test_pure_files": len(bundle.splits.test_pure),
            "train_samples": len(bundle.train_samples),
            "val_samples": len(bundle.val_samples),
            "test_samples": len(bundle.test_samples),
        },
        "signal_summary": {
            "impute_value": fill_value,
            "train_missing_rate": train_missing_rate,
            "val_missing_rate": val_missing_rate,
            "test_missing_rate": test_missing_rate,
            "train_confidence_mean": float(train_conf.mean()),
            "val_confidence_mean": float(val_conf.mean()),
            "test_confidence_mean": float(test_conf.mean()),
        },
        "threshold": threshold_info,
        "success_probability_threshold": success_threshold,
        "calibrator": calibrator.to_dict(),
        "metrics": {
            "train_soft_target": {
                "row_bce": float(
                    -np.mean(
                        y_train * np.log(np.clip(train_p, 1e-8, 1.0 - 1e-8))
                        + (1.0 - y_train) * np.log(np.clip(1.0 - train_p, 1e-8, 1.0 - 1e-8))
                    )
                ),
                "y_mean": float(y_train.mean()),
                "pred_mean": float(train_p.mean()),
            },
            "val_pure": val_metrics,
            "test_pure": test_metrics,
            "val_leaf_alarm": val_leaf_metrics,
            "test_leaf_alarm": test_leaf_metrics,
        },
    }

    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (output_dir / "calibrator.json").write_text(
        json.dumps(calibrator.to_dict(), indent=2),
        encoding="utf-8",
    )
    save_predictions_csv(output_dir / "val_predictions.csv", bundle.val_samples, val_conf, val_p, alarm_threshold)
    save_predictions_csv(output_dir / "test_predictions.csv", bundle.test_samples, test_conf, test_p, alarm_threshold)
    return summary


ROOT = Path.cwd()
DATA_DIR = ROOT / "tree_run_51284821"
OUTPUT_DIR = ROOT / "monitor_results" / "confidence_only_run"


## Run The Confidence-Only Baseline

This cell fits a one-dimensional logistic calibrator on the imputed
`confidence` values from mixed-root training prefixes, chooses the alarm
threshold on validation, then reports test metrics and leaf-level early
warning behavior.


In [2]:
summary = run_experiment(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    stride=8,
    min_step=9,
    pure_val_ratio=0.5,
    seed=7,
    calibration_learning_rate=0.1,
    calibration_epochs=400,
    calibration_l2=1e-3,
)

print("saved to:", OUTPUT_DIR)
print("alarm_score threshold:", summary["threshold"]["threshold"])
print("success_probability threshold:", summary["success_probability_threshold"])

metrics_table = pd.DataFrame([
    {"split": "val", **summary["metrics"]["val_pure"]},
    {"split": "test", **summary["metrics"]["test_pure"]},
])
display(metrics_table[[
    "split", "roc_auc", "average_precision", "balanced_accuracy",
    "precision", "recall", "specificity", "f1", "alarm_rate",
    "tp", "tn", "fp", "fn"
]])

leaf_table = pd.DataFrame([
    {"split": "val", **summary["metrics"]["val_leaf_alarm"]},
    {"split": "test", **summary["metrics"]["test_leaf_alarm"]},
])
display(leaf_table[[
    "split",
    "pure0_leaf_alarm_recall",
    "pure1_leaf_false_alarm_rate",
    "pure0_leaf_alarm_median_step",
    "pure1_leaf_false_alarm_median_step",
    "pure0_successful_warning_mean_early_pct",
    "pure0_leaf_paths",
    "pure1_leaf_paths",
]])

signal_table = pd.DataFrame([summary["signal_summary"]])
display(signal_table)


saved to: /mnt/d/Filez/Desktop/Research/raed/test/monitor_results/confidence_only_run
alarm_score threshold: 0.630253
success_probability threshold: 0.36974700000000005


,split,roc_auc,average_precision,balanced_accuracy,precision,recall,specificity,f1,alarm_rate,tp,tn,fp,fn
0,val,0.565344,0.876444,0.573873,0.897089,0.611727,0.536020,0.727422,0.592316,6625.0,878.0,760.0,4205.0
1,test,0.556374,0.886187,0.555723,0.897227,0.597371,0.514074,0.717220,0.583615,5727.0,694.0,656.0,3860.0


,split,pure0_leaf_alarm_recall,pure1_leaf_false_alarm_rate,pure0_leaf_alarm_median_step,pure1_leaf_false_alarm_median_step,pure0_successful_warning_mean_early_pct,pure0_leaf_paths,pure1_leaf_paths
0,val,0.999278,0.984749,9.0,16.0,80.332614,2771.0,459.0
1,test,0.997974,0.956298,9.0,16.0,77.377150,2468.0,389.0


,impute_value,train_missing_rate,val_missing_rate,test_missing_rate,train_confidence_mean,val_confidence_mean,test_confidence_mean
0,0.663775,0.009645,0.012191,0.012801,0.663775,0.670733,0.664374
